In [1]:
from pathlib import Path
import numpy as np
import os, json, random, pickle
from collections import Counter
from pathlib import Path

In [2]:
# Check image path

omama_dir_path = "/raid/mpsych/OMAMA/DATA/data/2d_resized_1024/images"
if not os.path.exists(omama_dir_path):
    print(f"File not found: {omama_dir_path}")
    print("Please update dicom_path with a valid image file path")
else:
    print(f"Found DICOM folder: {os.path.basename(omama_dir_path)}")
    omama_folder = Path("/raid/mpsych/OMAMA/DATA/data/2d_resized_1024/images")
    IDs = sorted(str(file) for file in omama_folder.rglob("*.npz"))
    print(len(IDs))

Found DICOM folder: images
163568


In [3]:
print(Path(IDs[0]).stem)

100000039159562031368112550794429920461


In [4]:
meta_dir = Path("/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/2d_resized_256/metadata")

In [5]:
# Get the headers
pkl = Path("/home/anya.tongprasith001/U54REC/release_to_header_mapping.pkl")
mapping = pickle.load(open(pkl, "rb"))
len(mapping)

163568

In [15]:
# Get the images, labels, and headers
noncancer = {}
cancer = {}

for ID in IDs :
    meta = json.load(open(meta_dir / f"{Path(ID).stem}.json"))
    if meta["label"] == "Unknown" :
        continue
        
    # Load image
    img_name = ID
    # Load window values
    wc = meta["WindowCenter"]
    wm = meta["WindowWidth"]
    window_center = float(wc[0] if hasattr(wc, '__len__') else wc)
    window_width = float(ww[0] if hasattr(ww, '__len__') else ww)
    window_min = window_center - window_width / 2
    window_max = window_center + window_width / 2
            
    if meta["label"] == "NonCancer" :
        if meta["PatientID"] not in noncancer.keys() :
            noncancer[meta["PatientID"]] = []
        noncancer[meta["PatientID"]].append([img_name, [window_min, window_max], 0])

    else : 
        if meta["PatientID"] not in cancer.keys() :
            cancer[meta["PatientID"]] = []
        cancer[meta["PatientID"]].append([img_name, [window_min, window_max], 1])

In [16]:
noncancer_ls = list(noncancer.items())
c = list(cancer.items())

print(len(noncancer_ls), len(c))

154238 3562


In [17]:
# Assign the images, metadata, and labels for setting up mix input in cancer cases
def assign_data(dataset) :
    images, metadata, labels = [],[],[]
    for patient, data_list in dataset :
        for data in data_list :
            #img = np.load(data[0])['data']
            #img = np.expand_dims(img, axis=-1)
            #images.append(img)
            images.append(data[0])
            metadata.append(data[1])
            labels.append(data[2])
    return images, metadata, labels

c_img, c_meta, c_labels = assign_data(c)
print(len(c_img), len(c_meta), len(c_labels))

7351 7351 7351


In [18]:
# Sample same numbers of non-cancer patients to create balanced dataset
count = 0
nc_groups = []
for i in range(10) :
    nc = noncancer_ls[count:count + len(c_img)]
    count += len(cancer.keys())
    nc_groups.append(nc)
nc_img, nc_meta, nc_labels = assign_data(nc_groups[0])
print(len(nc_img), len(nc_meta), len(nc_labels))

7351 7351 7351


In [19]:
# Split patients and merge cancer/non-cancer patients
train_size = int(0.7 * len(c_img))
val_size = int(0.15 * len(c_img))
test_size = int(0.15 * len(c_img))
print(train_size, val_size, test_size)
train_img = c_img[:train_size] + nc_img[:train_size]
train_meta = c_meta[:train_size] + nc_meta[:train_size]
train_labels = c_labels[:train_size] + nc_labels[:train_size]
val_img = c_img[train_size:train_size + val_size] + nc_img[train_size:train_size + val_size]
val_meta = c_meta[train_size:train_size + val_size] + nc_meta[train_size:train_size + val_size]
val_labels = c_labels[train_size:train_size + val_size] + nc_labels[train_size:train_size + val_size]
test_img = c_img[train_size + val_size:train_size + val_size + test_size] + nc_img[train_size + val_size:train_size + val_size + test_size]
test_meta = c_meta[train_size + val_size:train_size + val_size + test_size] + nc_meta[train_size + val_size:train_size + val_size + test_size]
test_labels = c_labels[train_size + val_size:train_size + val_size + test_size] + nc_labels[train_size + val_size:train_size + val_size + test_size]
print(f" Train : {len(train_img), len(train_meta), len(train_labels)}")
print(f" Val : {len(val_img), len(val_meta), len(val_labels)}")
print(f" Test : {len(test_img), len(test_meta), len(test_labels)}")


5145 1102 1102
 Train : (10290, 10290, 10290)
 Val : (2204, 2204, 2204)
 Test : (2204, 2204, 2204)


In [20]:
# Randomize the dataset
def shuffle_together(imgs, metadata, labels):
    combined = list(zip(imgs, metadata, labels))
    random.shuffle(combined)
    imgs, metadata, labels = zip(*combined)
    return list(imgs), list(metadata), list(labels)

train_imgs, train_metadata, train_labels = shuffle_together(train_img, train_meta, train_labels)
val_imgs, val_metadata, val_labels = shuffle_together(val_img, val_meta, val_labels)
test_imgs, test_metadata, test_labels = shuffle_together(test_img, test_meta, test_labels)

In [21]:
train_imgs = np.array(train_imgs)
train_metadata = np.array(train_metadata, dtype=np.float32)
train_labels = np.array(train_labels, dtype=np.float32)
val_imgs = np.array(val_imgs)
val_metadata = np.array(val_metadata, dtype=np.float32)
val_labels = np.array(val_labels, dtype=np.float32)
test_imgs = np.array(test_imgs)
test_metadata = np.array(test_metadata, dtype=np.float32)
test_labels = np.array(test_labels, dtype=np.float32)

In [22]:
# Check the distribution of classes
print(f'Average class probability in training set:   {train_labels.mean():.4f}')
print(f'Average class probability in validation set: {val_labels.mean():.4f}')
print(f'Average class probability in test set:       {test_labels.mean():.4f}')

Average class probability in training set:   0.5000
Average class probability in validation set: 0.5000
Average class probability in test set:       0.5000


In [23]:
# Save this training ds
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/train1c_imgs.npy', train_imgs)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/train1c_metadata.npy', train_metadata)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/train1c_labels.npy', train_labels)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/val1c_imgs.npy', val_imgs)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/val1c_metadata.npy', val_metadata)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/val1c_labels.npy', val_labels)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/test1c_imgs.npy', test_imgs)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/test1c_metadata.npy', test_metadata)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/test1c_labels.npy', test_labels)